# Saurabh Gupta
### ST 590 Final Project 

In [1]:
import pandas as pd
import os
import time
from pyspark.sql import SparkSession
from pyspark.sql.functions import mean, stddev, min, max, corr, median, count
from pyspark.sql.functions import col
from pyspark.ml.stat import Correlation
from pyspark.ml.feature import VectorAssembler
from pyspark.sql import functions as F
from pyspark.ml import Pipeline
from pyspark.ml.feature import VectorAssembler, OneHotEncoder, PCA, Binarizer
from pyspark.ml.regression import LinearRegression
from pyspark.ml.tuning import CrossValidator, ParamGridBuilder
from pyspark.ml.evaluation import RegressionEvaluator 
from pyspark.ml.feature import SQLTransformer, StringIndexer, Binarizer, VectorAssembler, VectorIndexer
from pyspark.sql.functions import explode, split
from pyspark.sql.types import StructType, StructField, DoubleType, LongType
from pyspark.sql.functions import input_file_name
import matplotlib.pyplot as plt


### 1 - Start the spark session and read in the data to create pyspark dataframe

In [2]:

spark = SparkSession.builder \
    .master('local[*]') \
    .appName("SaurabhSparkStream") \
    .getOrCreate()

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
24/04/30 01:25:37 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
24/04/30 01:25:38 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.


In [3]:
power_data = pd.read_csv("https://www4.stat.ncsu.edu/~online/datasets/power_ml_data.csv", sep = ",")
power_data.head(3)

,Temperature,Humidity,Wind_Speed,General_Diffuse_Flows,Diffuse_Flows,Power_Zone_1,Power_Zone_2,Power_Zone_3,Month,Hour
0,6.559,73.8,0.083,0.051,0.119,34055.69620,16128.87538,20240.96386,1,0
1,6.414,74.5,0.083,0.070,0.085,29814.68354,19375.07599,20131.08434,1,0
2,6.313,74.5,0.080,0.062,0.100,29128.10127,19006.68693,19668.43373,1,0


In [4]:
power_spark_data = spark.createDataFrame(power_data)
power_spark_data.show(3)

+-----------+--------+----------+---------------------+-------------+------------+------------+------------+-----+----+
|Temperature|Humidity|Wind_Speed|General_Diffuse_Flows|Diffuse_Flows|Power_Zone_1|Power_Zone_2|Power_Zone_3|Month|Hour|
+-----------+--------+----------+---------------------+-------------+------------+------------+------------+-----+----+
|      6.559|    73.8|     0.083|                0.051|        0.119|  34055.6962| 16128.87538| 20240.96386|    1|   0|
|      6.414|    74.5|     0.083|                 0.07|        0.085| 29814.68354| 19375.07599| 20131.08434|    1|   0|
|      6.313|    74.5|      0.08|                0.062|          0.1| 29128.10127| 19006.68693| 19668.43373|    1|   0|
+-----------+--------+----------+---------------------+-------------+------------+------------+------------+-----+----+
only showing top 3 rows



In [5]:
features = ["Temperature", "Humidity", "Wind_Speed", "General_Diffuse_Flows", "Diffuse_Flows", "Power_Zone_1", "Power_Zone_2", "Month", "Hour"]
response = "Power_Zone_3"
all_features = ["Temperature", "Humidity", "Wind_Speed", "General_Diffuse_Flows", "Diffuse_Flows", "Power_Zone_1", "Power_Zone_2", "Power_Zone_3", "Month", "Hour"]


### 2- Find means, standard deviations, min, max, and median values for all the variables where that makes sense

In [6]:
# Create an empty DataFrame to store the summary statistics
summary_stats_list = []

# Iterate over each feature and calculate summary statistics
for feature in features:
    # Filter the DataFrame to keep only the specified feature
    df_filtered = power_spark_data.select(feature)
    
    # Calculate summary statistics
    summary_stats = df_filtered.summary()
    summary_stats.show()


+-------+-----------------+
|summary|      Temperature|
+-------+-----------------+
|  count|            47174|
|   mean|18.81321980328147|
| stddev|5.813341359553924|
|    min|            3.247|
|    25%|            14.42|
|    50%|            18.78|
|    75%|            22.91|
|    max|            40.01|
+-------+-----------------+

+-------+------------------+
|summary|          Humidity|
+-------+------------------+
|  count|             47174|
|   mean|  68.2883982702336|
| stddev|15.560330479134226|
|    min|             11.34|
|    25%|             58.32|
|    50%|             69.89|
|    75%|              81.5|
|    max|              94.8|
+-------+------------------+

+-------+------------------+
|summary|        Wind_Speed|
+-------+------------------+
|  count|             47174|
|   mean|1.9616214016195355|
| stddev| 2.349351140467199|
|    min|              0.05|
|    25%|             0.078|
|    50%|             0.086|
|    75%|             4.915|
|    max|             6.

In [ ]:
# Calculate means, standard deviations, min, max, median for all numeric variables
# summary_df = power_spark_data.select([mean(col(c)).alias('mean_' + c) for c in features] +
#                              [stddev(col(c)).alias('stddev_' + c) for c in features] +
#                              [min(col(c)).alias('min_' + c) for c in features] +
#                              [max(col(c)).alias('max_' + c) for c in features] +
#                              [F.expr('percentile_approx(' + c + ', 0.5)').alias('median_' + c) for c in features])
# summary_df.show(3)

### 3- Find the correlations between all variables where that makes sense to do

In [8]:
# Find correlations between all numeric variables
# convert to vector column first
vector_col = "corr_features"
assembler = VectorAssembler(inputCols=power_spark_data.columns, outputCol=vector_col)
df_vector = assembler.transform(power_spark_data).select(vector_col)

# get correlation matrix
matrix = Correlation.corr(df_vector, vector_col)

### 4- Display correlation martrix

In [9]:
matrix.collect()[0]["pearson({})".format(vector_col)].values

array([ 1.00000000e+00, -4.60142941e-01,  4.76421096e-01,  4.59602100e-01,
        1.95625145e-01,  4.41446223e-01,  3.84301114e-01,  4.90752474e-01,
        2.84813779e-01,  1.99133704e-01, -4.60142941e-01,  1.00000000e+00,
       -1.36121474e-01, -4.67282484e-01, -2.58041806e-01, -2.89090125e-01,
       -2.97019392e-01, -2.34227664e-01, -1.67621115e-02, -2.44203512e-01,
        4.76421096e-01, -1.36121474e-01,  1.00000000e+00,  1.32303822e-01,
       -7.27182720e-04,  1.66322373e-01,  1.46337737e-01,  2.79112067e-01,
        1.68491442e-01,  4.28911867e-03,  4.59602100e-01, -4.67282484e-01,
        1.32303822e-01,  1.00000000e+00,  5.64530371e-01,  1.89993899e-01,
        1.58798487e-01,  6.49417432e-02, -2.07931061e-02,  1.31170662e-01,
        1.95625145e-01, -2.58041806e-01, -7.27182720e-04,  5.64530371e-01,
        1.00000000e+00,  8.28852283e-02,  4.73786964e-02, -3.67608724e-02,
       -1.30248930e-01,  1.32256967e-01,  4.41446223e-01, -2.89090125e-01,
        1.66322373e-01,  

In [ ]:
pip install Jinja2

In [10]:
matrix = Correlation.corr(df_vector, 'corr_features').collect()[0][0] 
corr_matrix = matrix.toArray().tolist() 
corr_matrix_df = pd.DataFrame(data=corr_matrix, columns = all_features, index=all_features) 
# Apply background gradient for visualization
styled_corr_matrix_df = corr_matrix_df.style.background_gradient(cmap='coolwarm')
# Display the styled DataFrame with specified precision
styled_corr_matrix_df_string = styled_corr_matrix_df.format('{:.2f}')
styled_corr_matrix_df_string

,Temperature,Humidity,Wind_Speed,General_Diffuse_Flows,Diffuse_Flows,Power_Zone_1,Power_Zone_2,Power_Zone_3,Month,Hour
Temperature,1.00,-0.46,0.48,0.46,0.20,0.44,0.38,0.49,0.28,0.20
Humidity,-0.46,1.00,-0.14,-0.47,-0.26,-0.29,-0.30,-0.23,-0.02,-0.24
Wind_Speed,0.48,-0.14,1.00,0.13,-0.00,0.17,0.15,0.28,0.17,0.00
General_Diffuse_Flows,0.46,-0.47,0.13,1.00,0.56,0.19,0.16,0.06,-0.02,0.13
Diffuse_Flows,0.20,-0.26,-0.00,0.56,1.00,0.08,0.05,-0.04,-0.13,0.13
Power_Zone_1,0.44,-0.29,0.17,0.19,0.08,1.00,0.83,0.75,-0.01,0.73
Power_Zone_2,0.38,-0.30,0.15,0.16,0.05,0.83,1.00,0.57,0.32,0.66
Power_Zone_3,0.49,-0.23,0.28,0.06,-0.04,0.75,0.57,1.00,-0.23,0.45
Month,0.28,-0.02,0.17,-0.02,-0.13,-0.01,0.32,-0.23,1.00,-0.00
Hour,0.20,-0.24,0.00,0.13,0.13,0.73,0.66,0.45,-0.00,1.00


### 5- Create a one-way contingency table of the Month variable. 

In [11]:
# Create one-way contingency table for Month variable
month_contingency = power_spark_data.groupBy('Month').agg(count('Month').alias('count'))

month_contingency.show(3)

+-----+-----+
|Month|count|
+-----+-----+
|    1| 4014|
|    2| 3588|
|    3| 4057|
+-----+-----+
only showing top 3 rows



### 6- Create a one-way contingency table of the Hour variable. 

In [12]:
# Create one-way contingency table for Hour variable
hour_contingency = power_spark_data.groupBy('Hour').agg(count('Hour').alias('count'))
hour_contingency.show(3)

+----+-----+
|Hour|count|
+----+-----+
|  19| 1950|
|   0| 1950|
|  22| 1966|
+----+-----+
only showing top 3 rows



### 7- Create a two-way contingency table for the Month and Hour variables

In [13]:
# Create two-way contingency table for Month and Hour variables
two_way_contingency = power_spark_data.crosstab('Month', 'Hour')
two_way_contingency.show(3)

24/04/30 01:26:21 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


+----------+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+
|Month_Hour|  0|  1| 10| 11| 12| 13| 14| 15| 16| 17| 18| 19|  2| 20| 21| 22| 23|  3|  4|  5|  6|  7|  8|  9|
+----------+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+
|         7|172|169|167|168|166|173|169|162|164|172|176|172|161|159|163|173|167|170|167|174|168|164|165|168|
|        11|159|168|160|160|156|158|162|163|157|166|165|160|168|161|162|159|164|154|165|153|162|167|164|164|
|         3|174|171|175|168|166|173|165|171|166|163|165|170|167|168|175|165|172|168|166|170|173|170|166|170|
+----------+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+
only showing top 3 rows



### 8 - Group the data by Month and find the means of the numeric variables

In [14]:
# Group data by Month and find means of numeric variables
grouped_means = power_spark_data.groupBy('Month').agg(*[mean(col(column)).alias('mean_' + column) for column in all_features])
grouped_means.show(3)

+-----+------------------+-----------------+------------------+--------------------------+------------------+-----------------+------------------+------------------+----------+------------------+
|Month|  mean_Temperature|    mean_Humidity|   mean_Wind_Speed|mean_General_Diffuse_Flows|mean_Diffuse_Flows|mean_Power_Zone_1| mean_Power_Zone_2| mean_Power_Zone_3|mean_Month|         mean_Hour|
+-----+------------------+-----------------+------------------+--------------------------+------------------+-----------------+------------------+------------------+----------+------------------+
|    1|12.734699053313403|68.25854758345788|0.7022234678624817|        103.95965819631294| 69.79882635774787|31052.98442786995|19407.916365649482| 17736.35168477329|       1.0| 11.51270553064275|
|    2|12.656535117056858|66.49092530657748|1.1139765886287636|        125.47113545150494| 92.33061510590859|30973.86315971572|18774.586005972407| 17309.70787041945|       2.0|11.497491638795987|
|    3|14.5840547202

### 9 - Group the data by Month and find the standard deviations of the numeric variables

In [15]:
# Group data by Month and find standard deviations of numeric variables
grouped_stddevs = power_spark_data.groupBy('Month').agg(*[stddev(col(column)).alias('stddev_' + column) for column in features])
grouped_stddevs.show(3)

+-----+------------------+------------------+------------------+----------------------------+--------------------+-------------------+-------------------+------------+-----------------+
|Month|stddev_Temperature|   stddev_Humidity| stddev_Wind_Speed|stddev_General_Diffuse_Flows|stddev_Diffuse_Flows|stddev_Power_Zone_1|stddev_Power_Zone_2|stddev_Month|      stddev_Hour|
+-----+------------------+------------------+------------------+----------------------------+--------------------+-------------------+-------------------+------------+-----------------+
|    1|3.2406352202253115|12.156169973955066|1.6117952014385528|          166.16470976772905|  131.45917158526208|  7402.323410683091|  4515.295696366689|         0.0|6.893386466701368|
|    2| 2.619715133289512|12.411941927246813|1.9811568615798596|          206.73018017500874|  169.15551710161597|  6874.584790780019|  4390.391100519847|         0.0|6.900280948950496|
|    3| 3.758851761909596|13.918146099336548|1.9009817904056054|      

In [ ]:
# Show the summaries
#summary_df.show(3)


### 10 - The Hour column is likely not stored as a DoubleType. Use a spark SQL data frame method to cast the variable as a DoubleType


In [16]:
# Cast Hour column to DoubleType
spark_df = power_spark_data.withColumn("Hour_double", power_spark_data["Hour"].cast("double"))

# Verify the schema to ensure the casting was successful
spark_df.printSchema()

root
 |-- Temperature: double (nullable = true)
 |-- Humidity: double (nullable = true)
 |-- Wind_Speed: double (nullable = true)
 |-- General_Diffuse_Flows: double (nullable = true)
 |-- Diffuse_Flows: double (nullable = true)
 |-- Power_Zone_1: double (nullable = true)
 |-- Power_Zone_2: double (nullable = true)
 |-- Power_Zone_3: double (nullable = true)
 |-- Month: long (nullable = true)
 |-- Hour: long (nullable = true)
 |-- Hour_double: double (nullable = true)



### 11- Binarize the Hour column based on the column being less than 6.5 or not (night vs day essentially)

In [17]:
# Binarize the Hour column
binaryTrans = Binarizer(threshold = 6.5, inputCol = "Hour_double", outputCol = "Hour_binary")
binaryTrans.transform(spark_df).show(3)

+-----------+--------+----------+---------------------+-------------+------------+------------+------------+-----+----+-----------+-----------+
|Temperature|Humidity|Wind_Speed|General_Diffuse_Flows|Diffuse_Flows|Power_Zone_1|Power_Zone_2|Power_Zone_3|Month|Hour|Hour_double|Hour_binary|
+-----------+--------+----------+---------------------+-------------+------------+------------+------------+-----+----+-----------+-----------+
|      6.559|    73.8|     0.083|                0.051|        0.119|  34055.6962| 16128.87538| 20240.96386|    1|   0|        0.0|        0.0|
|      6.414|    74.5|     0.083|                 0.07|        0.085| 29814.68354| 19375.07599| 20131.08434|    1|   0|        0.0|        0.0|
|      6.313|    74.5|      0.08|                0.062|          0.1| 29128.10127| 19006.68693| 19668.43373|    1|   0|        0.0|        0.0|
+-----------+--------+----------+---------------------+-------------+------------+------------+------------+-----+----+-----------+-----

### 12 - SQL transformer for standarize 

In [18]:
sqlTrans = SQLTransformer(
    statement = """
                SELECT 
                log(Temperature) as log_Temperature, 
                log(Humidity) as log_Humidity, 
                log(Wind_Speed) as log_Wind_Speed, 
                log(General_Diffuse_Flows) as log_General_Diffuse_Flows, 
                log(Diffuse_Flows) as log_Diffuse_Flows, 
                log(Power_Zone_1) as log_Power_Zone_1, 
                log(Power_Zone_2) as log_Power_Zone_2, 
                Month as Month, 
                Hour as Hour, 
                Hour_double as Hour_double, 
                Power_Zone_3 as label 
                FROM __THIS__
                """
)

In [19]:
sqlTrans.transform(
    binaryTrans.transform(spark_df)
).show(3)

+------------------+-----------------+-------------------+-------------------------+-------------------+------------------+-----------------+-----+----+-----------+-----------+
|   log_Temperature|     log_Humidity|     log_Wind_Speed|log_General_Diffuse_Flows|  log_Diffuse_Flows|  log_Power_Zone_1| log_Power_Zone_2|Month|Hour|Hour_double|      label|
+------------------+-----------------+-------------------+-------------------------+-------------------+------------------+-----------------+-----+----+-----------+-----------+
|1.8808381523115985|4.301358731606427| -2.488914671185539|      -2.9759296462578115|-2.1286317858706076|10.435752586866837| 9.68836644643188|    1|   0|        0.0|20240.96386|
| 1.858483101270963|4.310799125385514| -2.488914671185539|       -2.659260036932778|-2.4651040224918206|10.302756287380467|9.871742776278372|    1|   0|        0.0|20131.08434|
|1.8426109993854884|4.310799125385514|-2.5257286443082556|      -2.7806208939370456|-2.3025850929940455|10.27945866

### 13- One-hot encode the Month column

In [20]:
# One-hot encode the Month column
encoder = OneHotEncoder(inputCol="Month", outputCol="Month_encoded")


### 14- Run a PCA fit on the Temperature, Humidity, Wind_Speed, General_Diffuse_Flows, and Diffuse_Flows columns

In [21]:
# Assemble features for PCA
assembler = VectorAssembler(inputCols=["log_Temperature", "log_Humidity", "log_Wind_Speed", 
                                           "log_General_Diffuse_Flows", "log_Diffuse_Flows"], 
                                outputCol="features")



In [22]:
# Run PCA fit
pca = PCA(k=3, inputCol="features", outputCol="pca_features")


### 15-  For your elastic net model you’ll use the fitted PCA features, binary Hour variable, Power_Zone_1, Power_Zone_2, and Month indicator variables as your predictors.

In [23]:
# Assemble final features
assembler_final = VectorAssembler(inputCols=["pca_features", "Hour_binary", 
                                             "log_Power_Zone_1", "log_Power_Zone_2", 
                                             "Month_encoded"], 
                                  outputCol="features_final")

### 16 - You should rename the Power_Zone_3 variable appropriately to be your response.

In [24]:
# Rename the Power_Zone_3 variable
spark_df_new = spark_df.withColumnRenamed("Power_Zone_3", "label")

In [25]:
# Define ElasticNet model
lr_model = LinearRegression(solver="normal", featuresCol="features_final", labelCol="label")
# Define the evaluator
evaluator = RegressionEvaluator(predictionCol="prediction", labelCol="label", metricName="rmse")


### 17 -  This ends the pipeline of transformations.

In [26]:
# Define parameter grid for cross-validation
paramGrid = ParamGridBuilder() \
    .addGrid(lr_model.regParam, [ 0, 0.05, 0.1, 0.25, 0.5, 0.75, 0.9, 0.95, 0.98, 0.99, 1]) \
    .addGrid(lr_model.elasticNetParam, [ 0, 0.05, 0.1, 0.25, 0.5, 0.75, 0.9, 0.95, 0.98, 0.99, 1]) \
    .build()

In [27]:
spark_df.show(3)

+-----------+--------+----------+---------------------+-------------+------------+------------+------------+-----+----+-----------+
|Temperature|Humidity|Wind_Speed|General_Diffuse_Flows|Diffuse_Flows|Power_Zone_1|Power_Zone_2|Power_Zone_3|Month|Hour|Hour_double|
+-----------+--------+----------+---------------------+-------------+------------+------------+------------+-----+----+-----------+
|      6.559|    73.8|     0.083|                0.051|        0.119|  34055.6962| 16128.87538| 20240.96386|    1|   0|        0.0|
|      6.414|    74.5|     0.083|                 0.07|        0.085| 29814.68354| 19375.07599| 20131.08434|    1|   0|        0.0|
|      6.313|    74.5|      0.08|                0.062|          0.1| 29128.10127| 19006.68693| 19668.43373|    1|   0|        0.0|
+-----------+--------+----------+---------------------+-------------+------------+------------+------------+-----+----+-----------+
only showing top 3 rows



In [28]:
# Construct pipeline
#pipeline = Pipeline(stages=[binaryTrans, sqlTrans, encoder, assembler, pca, assembler_final, crossval])

pipeline = Pipeline(stages=[ sqlTrans,binaryTrans, encoder, assembler, pca, assembler_final, lr_model])


In [29]:
# Define cross-validator
crossval = CrossValidator(estimator=pipeline,
                          estimatorParamMaps=paramGrid,
                          evaluator=evaluator,
                          numFolds=5)

### 18 -  Fit the model!

In [30]:
# Fit the pipeline
sparkModel = crossval.fit(spark_df)

24/04/30 01:27:03 WARN Instrumentation: [39783c8e] regParam is zero, which might cause numerical instability and overfitting.
24/04/30 01:27:03 WARN Instrumentation: [39783c8e] Cholesky solver failed due to singular covariance matrix. Retrying with Quasi-Newton solver.
24/04/30 01:27:04 ERROR LBFGS: Failure! Resetting history: breeze.optimize.FirstOrderException: Line search zoom failed
24/04/30 01:27:04 ERROR LBFGS: Failure! Resetting history: breeze.optimize.FirstOrderException: Line search zoom failed
24/04/30 01:27:04 ERROR LBFGS: Failure again! Giving up and returning. Maybe the objective is just poorly behaved?
24/04/30 01:27:06 WARN Instrumentation: [41b3a8f8] regParam is zero, which might cause numerical instability and overfitting.
24/04/30 01:27:06 WARN Instrumentation: [41b3a8f8] Cholesky solver failed due to singular covariance matrix. Retrying with Quasi-Newton solver.
24/04/30 01:27:06 ERROR LBFGS: Failure! Resetting history: breeze.optimize.FirstOrderException: Line sear

In [31]:
best_model = sparkModel.bestModel

In [32]:
# Print the best model's parameters
print("Best model parameters:")
print("regParam:", best_model.stages[-1].getOrDefault("regParam"))
print("elasticNetParam:", best_model.stages[-1].getOrDefault("elasticNetParam"))

Best model parameters:
regParam: 0.9
elasticNetParam: 0.25


### 19 -  Find the training set RMSE as done in the notes by using your fitted model as a transformer.

In [33]:
# Make predictions on the training data
predictions = sparkModel.transform(spark_df)
#predictions.show(3)

In [34]:
# Compute RMSE
evaluator = RegressionEvaluator(labelCol="label", predictionCol="prediction", metricName="rmse")
rmse = evaluator.evaluate(predictions)
print("Root Mean Squared Error (RMSE) on training data: {:.2f}".format(rmse))

Root Mean Squared Error (RMSE) on training data: 2205.69


### 20 -  Take the outputted transformations from the model (the predictions) and create a residual column (label - prediction). The .withColumn() method is handy.

In [35]:
# Create residual column
predictions = predictions.withColumn("residual", col("label") - col("prediction"))
#predictions1.show()
# Show predictions and residuals
pred = predictions.select("label", "prediction", "residual")
pred.show()

+-----------+------------------+------------------+
|      label|        prediction|          residual|
+-----------+------------------+------------------+
|20240.96386|21064.350389666564|-823.3865296665645|
|20131.08434|19006.158381466317| 1124.925958533684|
|19668.43373|18586.702838863217|1081.7308911367836|
|18899.27711|17836.615549137263|1062.6615608627362|
|18442.40964|17430.421297505207|1011.9883424947948|
|18130.12048|16832.402516314352| 1297.717963685649|
|17945.06024|16435.358617356716|1509.7016226432825|
|17459.27711|15973.224021032132|1486.0530889678666|
|17025.54217|  15366.0019901809|1659.5401798191015|
|16794.21687|15001.720742204314|1792.4961277956863|
|16638.07229| 14719.39387256783|1918.6784174321692|
|16395.18072|14371.878939394112|2023.3017806058888|
|16117.59036|13912.586600155395| 2205.003759844605|
| 15822.6506|13309.516949628189| 2513.133650371812|
|15672.28916|13153.879486351769|2518.4096736482315|
|15597.10843|12896.379172590707|2700.7292574092935|
|15510.36145

# Streaming Part 

## Produce Data (10 

In [36]:
# Display information about the current Spark session
print("Spark version:", spark.version)
print("Spark master:", spark.sparkContext.master)
print("Application name:", spark.sparkContext.appName)

Spark version: 3.5.0
Spark master: local[*]
Application name: SaurabhSparkStream


## Reading a Stream (15 pts) 

### Read in a stream in the form of .csv files. Create a folder to send.csv files.

### Setup the schema for the stream 

In [37]:
# Define the schema
myschema = StructType([
    StructField("Temperature", DoubleType(), nullable=True),
    StructField("Humidity", DoubleType(), nullable=True),
    StructField("Wind_Speed", DoubleType(), nullable=True),
    StructField("General_Diffuse_Flows", DoubleType(), nullable=True),
    StructField("Diffuse_Flows", DoubleType(), nullable=True),
    StructField("Power_Zone_1", DoubleType(), nullable=True),
    StructField("Power_Zone_2", DoubleType(), nullable=True),
    StructField("Power_Zone_3", DoubleType(), nullable=True),
    StructField("Month", LongType(), nullable=True),
    StructField("Hour", LongType(), nullable=True),
    StructField("Hour_double", DoubleType(), nullable=True)
])

In [ ]:
#streaming_memory_df = streaming_df.writeStream.format("memory").queryName("my_stream").start()
#mydf = spark.sql("select * from my_stream").show() 
#df = spark.readStream.schema(myschema).csv("csv_files", header=True)
# df = spark \
#     .readStream \
#     .schema(myschema) \
#     .option("cleanSource", "archive") \
#     .option("sourceArchiveDir","/home/archive") \
#     .csv("csv_files")


# Define a custom function to archive the file after processing each micro-batch
# def archive_file(df, epoch_id):
#     if epoch_id == 0:
#         shutil.move("csv_files/*", "csv_files_old/*")

# query = streaming_df.writeStream \
#     .foreachBatch(archive_file) \
#     .start()

# Wait for the stream to finish
#query.awaitTermination()
    


### Set up the readStream.

In [38]:
streaming_df = spark.readStream.schema(myschema).option("header", "true").csv("csv_files") 


In [39]:
query_data = streaming_df.writeStream \
    .outputMode("append") \
    .format("console") \
    .start()


24/04/30 01:43:12 WARN ResolveWriteToStream: Temporary checkpoint location created which is deleted normally when the query didn't fail: /tmp/temporary-de987120-bf7b-486b-9d57-6eafacabdbf4. If it's required to delete it under any circumstances, please set spark.sql.streaming.forceDeleteTempCheckpointLocation to true. Important to know deleting temp checkpoint folder is best effort.
24/04/30 01:43:12 WARN ResolveWriteToStream: spark.sql.adaptive.enabled is not supported in streaming DataFrames/Datasets and will be disabled.


In [ ]:
#rateDF = spark.readStream.schema(myschema).csv("csv_files")

In [ ]:
#writeDF = rateDF.writeStream.outputMode("append").format("console").start()

In [ ]:
#writeDF.stop()

## Transform/Aggregation Step (20 pts)

In [40]:
#pipeline_stream_model = pipeline1.fit(streaming_df)
predictions = sparkModel.transform(streaming_df)
# Create residual column
predictions = predictions.withColumn("residual", col("label") - col("prediction"))
# Show predictions and residuals
predictions_temp = predictions.select("label", "prediction", "residual")


### With your stream, use your model transformer to obtain predictions from the incoming data

In [41]:
query_predictions = predictions_temp.writeStream \
    .outputMode("append") \
    .format("console") \
    .start()

24/04/30 01:43:16 WARN ResolveWriteToStream: Temporary checkpoint location created which is deleted normally when the query didn't fail: /tmp/temporary-d1825838-f66f-42dd-921b-02fa1e72a98e. If it's required to delete it under any circumstances, please set spark.sql.streaming.forceDeleteTempCheckpointLocation to true. Important to know deleting temp checkpoint folder is best effort.
24/04/30 01:43:16 WARN ResolveWriteToStream: spark.sql.adaptive.enabled is not supported in streaming DataFrames/Datasets and will be disabled.


### create a second pipeline

In [42]:
predictions_all = sparkModel.transform(streaming_df)
#predictions_all = predictions_all.withColumn("residual", col("label") - col("prediction"))
# Show predictions and residuals


In [ ]:
# query_predictions_all = predictions_all.writeStream \
#     .outputMode("append") \
#     .format("console") \
#     .start()

### Now join these two data frames together based on the label variable 

In [43]:
#Set up the join and start the query!
joinquery = predictions_all \
                .join(predictions_temp, "label", "inner") \
                .writeStream.outputMode("append") \
                .format("console") \
                .start()

24/04/30 01:43:26 WARN ResolveWriteToStream: Temporary checkpoint location created which is deleted normally when the query didn't fail: /tmp/temporary-30b29d04-05be-4035-94bd-0d290d6cb158. If it's required to delete it under any circumstances, please set spark.sql.streaming.forceDeleteTempCheckpointLocation to true. Important to know deleting temp checkpoint folder is best effort.
24/04/30 01:43:26 WARN ResolveWriteToStream: spark.sql.adaptive.enabled is not supported in streaming DataFrames/Datasets and will be disabled.


-------------------------------------------
Batch: 0
-------------------------------------------
-------------------------------------------
Batch: 0
-------------------------------------------
+-----------+------------------+-------------------+
|      label|        prediction|           residual|
+-----------+------------------+-------------------+
|16936.03239| 17937.06690865601| -1001.034518656008|
|11076.92308|10042.987363369059|  1033.935716630942|
|27138.80878| 26040.62888331135|  1098.179896688649|
|21547.33542|22850.878171350807|-1303.5427513508075|
|15166.38655|17689.182107151748|-2522.7955571517487|
+-----------+------------------+-------------------+

+-----------+--------+----------+---------------------+-------------+------------+------------+------------+-----+----+-----------+
|Temperature|Humidity|Wind_Speed|General_Diffuse_Flows|Diffuse_Flows|Power_Zone_1|Power_Zone_2|Power_Zone_3|Month|Hour|Hour_double|
+-----------+--------+----------+---------------------+---------

-------------------------------------------
Batch: 0
-------------------------------------------
-------------------------------------------
Batch: 1
-------------------------------------------
+-----------+--------+----------+---------------------+-------------+------------+------------+------------+-----+----+-----------+
|Temperature|Humidity|Wind_Speed|General_Diffuse_Flows|Diffuse_Flows|Power_Zone_1|Power_Zone_2|Power_Zone_3|Month|Hour|Hour_double|
+-----------+--------+----------+---------------------+-------------+------------+------------+------------+-----+----+-----------+
|      24.13|    83.9|     4.928|                 0.08|        0.115| 48100.15538| 30103.90707| 31803.38558|    8|  20|       20.0|
|      10.07|    74.3|     0.083|                0.073|        0.111| 22177.21519| 13641.33739| 14093.49398|    1|   5|        5.0|
|      22.57|   54.82|     4.914|                134.3|        143.4| 37746.38298| 19251.21951| 18789.67742|    3|  18|       18.0|
|      12.01| 

-------------------------------------------
Batch: 2
-------------------------------------------
+-----------+--------+----------+---------------------+-------------+------------+------------+------------+-----+----+-----------+
|Temperature|Humidity|Wind_Speed|General_Diffuse_Flows|Diffuse_Flows|Power_Zone_1|Power_Zone_2|Power_Zone_3|Month|Hour|Hour_double|
+-----------+--------+----------+---------------------+-------------+------------+------------+------------+-----+----+-----------+
|      23.57|   49.99|     0.082|                 81.9|         94.1| 41998.97872| 22145.12195| 26221.93548|    3|  18|       18.0|
|      21.12|    85.1|      0.22|                0.062|        0.126| 23078.23009| 14246.56965|  11208.1716|    9|   5|        5.0|
|      22.61|    67.2|     0.073|                862.0|        335.0| 30808.13115| 17405.57276|  16055.7085|    5|  12|       12.0|
|      23.16|    71.8|     0.072|                729.0|        508.4| 33708.07947| 18722.24532| 15531.32308|   

-------------------------------------------
Batch: 1
-------------------------------------------
+-----------+------------------+------------------+------------------+-------------------------+-------------------+------------------+------------------+-----+----+-----------+-----------+---------------+--------------------+--------------------+--------------------+------------------+------------------+-------------------+
|      label|   log_Temperature|      log_Humidity|    log_Wind_Speed|log_General_Diffuse_Flows|  log_Diffuse_Flows|  log_Power_Zone_1|  log_Power_Zone_2|Month|Hour|Hour_double|Hour_binary|  Month_encoded|            features|        pca_features|      features_final|        prediction|        prediction|           residual|
+-----------+------------------+------------------+------------------+-------------------------+-------------------+------------------+------------------+-----+----+-----------+-----------+---------------+--------------------+--------------------+--

...

-------------------------------------------
Batch: 16
-------------------------------------------
+-----------+------------------+------------------+-------------------+-------------------------+-------------------+------------------+------------------+-----+----+-----------+-----------+---------------+--------------------+--------------------+--------------------+------------------+------------------+-------------------+
|      label|   log_Temperature|      log_Humidity|     log_Wind_Speed|log_General_Diffuse_Flows|  log_Diffuse_Flows|  log_Power_Zone_1|  log_Power_Zone_2|Month|Hour|Hour_double|Hour_binary|  Month_encoded|            features|        pca_features|      features_final|        prediction|        prediction|           residual|
+-----------+------------------+------------------+-------------------+-------------------------+-------------------+------------------+------------------+-----+----+-----------+-----------+---------------+--------------------+-------------------

-------------------------------------------
Batch: 25
-------------------------------------------
-------------------------------------------
Batch: 25
-------------------------------------------
+-----------+--------+----------+---------------------+-------------+------------+------------+------------+-----+----+-----------+
|Temperature|Humidity|Wind_Speed|General_Diffuse_Flows|Diffuse_Flows|Power_Zone_1|Power_Zone_2|Power_Zone_3|Month|Hour|Hour_double|
+-----------+--------+----------+---------------------+-------------+------------+------------+------------+-----+----+-----------+
|      15.21|    89.1|     0.072|                0.062|        0.156| 21661.53846| 17152.06612| 11051.56627|   11|   2|        2.0|
|      20.44|   62.82|     0.075|                0.077|        0.107| 45418.80795| 24537.62994| 27518.03077|    6|  21|       21.0|
|      13.99|    89.8|     0.067|                117.0|        105.3| 34107.38428| 19444.39919| 19601.45455|    4|  12|       12.0|
|      19.12

-------------------------------------------
Batch: 26
-------------------------------------------
+-----------+--------+----------+---------------------+-------------+------------+------------+------------+-----+----+-----------+
|Temperature|Humidity|Wind_Speed|General_Diffuse_Flows|Diffuse_Flows|Power_Zone_1|Power_Zone_2|Power_Zone_3|Month|Hour|Hour_double|
+-----------+--------+----------+---------------------+-------------+------------+------------+------------+-----+----+-----------+
|       10.9|    81.2|     0.079|                0.044|        0.104| 22332.30769| 17739.66942| 9466.987952|   11|   4|        4.0|
|      18.82|    72.1|      0.07|                307.2|        292.9| 25016.65574| 11565.32508|  12283.7247|    5|   8|        8.0|
|      19.33|    83.0|     4.917|                0.055|        0.193| 26190.98468| 16771.36929| 11473.31307|   10|   1|        1.0|
|      13.72|   46.64|     0.086|                 97.9|        101.0| 35246.58228| 22537.38602| 16707.46988|  

-------------------------------------------
Batch: 17
-------------------------------------------
+-----------+------------------+-----------------+------------------+-------------------------+------------------+------------------+------------------+-----+----+-----------+-----------+---------------+--------------------+--------------------+--------------------+------------------+------------------+-------------------+
|      label|   log_Temperature|     log_Humidity|    log_Wind_Speed|log_General_Diffuse_Flows| log_Diffuse_Flows|  log_Power_Zone_1|  log_Power_Zone_2|Month|Hour|Hour_double|Hour_binary|  Month_encoded|            features|        pca_features|      features_final|        prediction|        prediction|           residual|
+-----------+------------------+-----------------+------------------+-------------------------+------------------+------------------+------------------+-----+----+-----------+-----------+---------------+--------------------+--------------------+-------

-------------------------------------------
Batch: 27
-------------------------------------------
+-----------+--------+----------+---------------------+-------------+------------+------------+------------+-----+----+-----------+
|Temperature|Humidity|Wind_Speed|General_Diffuse_Flows|Diffuse_Flows|Power_Zone_1|Power_Zone_2|Power_Zone_3|Month|Hour|Hour_double|
+-----------+--------+----------+---------------------+-------------+------------+------------+------------+-----+----+-----------+
|       27.2|   47.22|     4.933|                135.5|        149.1| 39757.42508| 23230.83421| 19729.65517|    8|  18|       18.0|
|      13.07|    84.2|     0.075|                0.066|        0.126| 25482.53165| 15961.09422| 12913.73494|    1|   7|        7.0|
|      21.57|    77.4|     0.069|                388.5|        366.8| 25697.48344| 16809.97921| 11260.06154|    6|   8|        8.0|
|      15.92|    76.0|     0.073|                0.059|        0.056| 38018.46154| 32188.01653| 19998.07229|  

### STOP all queries execution 

In [ ]:

#stop query
query_data.awaitTermination()
query_data.stop()


In [ ]:
query_predictions.awaitTermination()
query_predictions.stop()

In [ ]:
# query_predictions_all.awaitTermination()
# query_predictions_all.stop()

In [ ]:
joinquery.stop()

In [ ]:
# NOT REQUIRED FOR PROJECT. 
######################################
# Test script to split main data file into small subsets.
power_data_stream = pd.read_csv("data/power_streaming_data.csv", sep = ",")
power_data_stream.head(3)
# Define the maximum number of rows per file
rows_per_file = 500  # Change this value as needed
# Calculate the number of files needed
num_files = len(power_data_stream) // rows_per_file + 1
# Define the output folder
output_folder = "csv_files_old"  # Change this to your desired output folder
# Create the output folder if it doesn't exist
if not os.path.exists(output_folder):
    os.makedirs(output_folder)
# Iterate over the dataset and write subsets to separate CSV files
for i in range(num_files):
    start_index = i * rows_per_file
    end_index = (i + 1) * rows_per_file
    subset = power_data_stream.iloc[start_index:end_index]
    subset.to_csv(os.path.join(output_folder,f"power_streaming_data_subset_{i + 1}.csv"), index=False)
    print(f"Subset {i + 1} written to power_streaming_data_subset_{i + 1}.csv")